# Serve Llama 3.1 8B Instruct for the refund assistant (Kaggle GPU)

This notebook starts an OpenAI-compatible vLLM server and exposes it through a
cloudflared quick tunnel. The app on Mri's machine talks to it through the
existing `openai_compatible` provider.

**Before running**

1. Notebook settings: **Accelerator = GPU T4 x2** (preferred) or **GPU T4 x1**;
   **Internet = On** (requires a phone-verified Kaggle account).
2. Accept Meta's license for the Llama model:
   - on Hugging Face with your account, and create a **new read-only** token
     after revoking the old exposed one; store it as the Kaggle Secret
     `HF_TOKEN`; **or**
   - attach the model from Kaggle Models to this notebook and set
     `MODEL_SOURCE = "kaggle_model"` below.
3. Add Kaggle Secrets: `API_KEY` (any random string you choose; the app must use
   the same value) and `HF_TOKEN` (only for the Hugging Face route).
   Optional `NGROK_TOKEN` only if you switch the tunnel to ngrok.
4. Never paste a token into a cell. This notebook reads secrets only.

**Memory rule:** 8B in fp16 is ~16 GB of weights and does not fit one 16 GB T4.
- two T4s: fp16, `--tensor-parallel-size 2`
- one T4: a 4-bit AWQ build, `--max-model-len 8192`

In [ ]:
# Cell 1 - settings and versions.  Change MODEL_ID only after checking the repo
# on Hugging Face (gated repos need the license accepted first).
MODEL_SOURCE = "huggingface"          # "huggingface" or "kaggle_model"
HF_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
AWQ_MODEL_ID = "casperhansen/llama-3.1-8b-instruct-awq"
KAGGLE_MODEL_PATH = ""                # set when MODEL_SOURCE == "kaggle_model"
SERVED_NAME = "refund-llm"
MAX_MODEL_LEN = 8192
PORT = 8000
VLLM_SPEC = "vllm"                    # pin only after a working install; log the version

print("MODEL_SOURCE =", MODEL_SOURCE)
print("vLLM spec    =", VLLM_SPEC)

In [ ]:
# Cell 2 - install vLLM.  If the pinned/selected build fails, the exact pip
# error is shown; report it instead of guessing.
import subprocess, sys, textwrap

def run(cmd, check=True):
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout[-4000:])
    if proc.stderr:
        print(proc.stderr[-4000:])
    if check and proc.returncode != 0:
        raise SystemExit(f"command failed with {proc.returncode}: {' '.join(cmd)}")
    return proc

run([sys.executable, "-m", "pip", "install", "-q", VLLM_SPEC])
import vllm
print("vllm version:", vllm.__version__)

In [ ]:
# Cell 3 - read secrets.  Nothing here is printed.
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def optional_secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return None

API_KEY = optional_secret("API_KEY")
if not API_KEY:
    raise SystemExit("Missing Kaggle Secret API_KEY. Add it in Add-ons > Secrets.")

HF_TOKEN = optional_secret("HF_TOKEN") if MODEL_SOURCE == "huggingface" else None
NGROK_TOKEN = optional_secret("NGROK_TOKEN")
print("API_KEY loaded:", bool(API_KEY))
print("HF_TOKEN loaded:", bool(HF_TOKEN))
print("NGROK_TOKEN loaded:", bool(NGROK_TOKEN))

In [ ]:
# Cell 4 - inspect GPUs and choose dtype/quantization.
import subprocess

smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print(smi.stdout.strip() or smi.stderr.strip())
GPU_COUNT = max(1, smi.stdout.count("GPU "))

if MODEL_SOURCE == "kaggle_model":
    if not KAGGLE_MODEL_PATH:
        raise SystemExit("Set KAGGLE_MODEL_PATH to the attached Kaggle Models path.")
    MODEL = KAGGLE_MODEL_PATH
elif GPU_COUNT >= 2:
    MODEL = HF_MODEL_ID
    print("Two or more GPUs: using fp16 with tensor parallelism.")
else:
    MODEL = AWQ_MODEL_ID
    print("One GPU detected: using the 4-bit AWQ build to fit 16 GB.")

if GPU_COUNT >= 2:
    TENSOR_PARALLEL = 2
    QUANTIZATION = None
else:
    TENSOR_PARALLEL = 1
    if MODEL == AWQ_MODEL_ID:
        QUANTIZATION = "awq"
    else:
        raise SystemExit(
            "One 16 GB GPU cannot hold fp16 8B weights (~16 GB). "
            "Use a T4 x2 session, or point MODEL at a 4-bit AWQ build."
        )
print("model:", MODEL)
print("tensor_parallel:", TENSOR_PARALLEL, "quantization:", QUANTIZATION)


In [ ]:
# Cell 5 - launch the vLLM OpenAI server in the background.
import os, subprocess, time

LOG_PATH = "/kaggle/working/vllm.log"
cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    "--served-model-name", SERVED_NAME,
    "--dtype", "half",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--port", str(PORT),
    "--api-key", API_KEY,
    "--trust-remote-code",
]
if QUANTIZATION:
    cmd += ["--quantization", QUANTIZATION]
if TENSOR_PARALLEL > 1:
    cmd += ["--tensor-parallel-size", str(TENSOR_PARALLEL)]

env = dict(os.environ)
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN          # environment only, never on the command line
    env["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

log_handle = open(LOG_PATH, "w")
server = subprocess.Popen(cmd, stdout=log_handle, stderr=subprocess.STDOUT, env=env)
print("server pid:", server.pid, "log:", LOG_PATH)
time.sleep(5)
print(open(LOG_PATH).read()[-2000:])

In [ ]:
# Cell 6 - wait for /v1/models (cold start can take several minutes).
import requests, time

base = f"http://127.0.0.1:{PORT}/v1"
headers = {"Authorization": f"Bearer {API_KEY}"}
deadline = time.time() + 10 * 60
ready = False
while time.time() < deadline:
    if server.poll() is not None:
        print("server exited with", server.returncode)
        print(open(LOG_PATH).read()[-6000:])
        raise SystemExit("vLLM died. If the log shows 401/403: license not accepted or HF_TOKEN wrong.")
    try:
        r = requests.get(f"{base}/models", headers=headers, timeout=5)
        if r.status_code == 200:
            ready = True
            print("server ready:", r.json())
            break
    except Exception:
        pass
    time.sleep(10)

if not ready:
    print(open(LOG_PATH).read()[-8000:])
    raise SystemExit("server not ready after 10 minutes")

In [ ]:
# Cell 7 - prove JSON-schema constrained decoding works for the app.
import json, requests

schema = {
    "type": "object",
    "properties": {
        "grades": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "provision_id": {"type": "string"},
                    "grade": {"type": "integer", "minimum": 0, "maximum": 3},
                    "reason": {"type": "string"},
                },
                "required": ["provision_id", "grade", "reason"],
            },
        }
    },
    "required": ["grades"],
}
payload = {
    "model": SERVED_NAME,
    "messages": [{"role": "user", "content": "Grade provision P1 as relevant. Return JSON."}],
    "temperature": 0,
    "max_tokens": 200,
    "response_format": {
        "type": "json_schema",
        "json_schema": {"name": "RelevanceGradeBatch", "schema": schema},
    },
}
r = requests.post(f"{base}/chat/completions", headers=headers, json=payload, timeout=120)
print("status:", r.status_code)
try:
    content = r.json()["choices"][0]["message"]["content"]
    print("content:", content)
    parsed = json.loads(content)
    print("JSON schema test:", "PASS" if "grades" in parsed else "FAIL")
except Exception as exc:
    print("JSON schema test: FAIL", type(exc).__name__, exc)
    print(r.text[:2000])

In [ ]:
# Cell 8 - open a cloudflared quick tunnel (no account needed).
import re, subprocess, time

install = subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64 -O /usr/local/bin/cloudflared "
    "&& chmod +x /usr/local/bin/cloudflared",
    shell=True, capture_output=True, text=True,
)
print(install.stderr[-1000:])
subprocess.run(["/usr/local/bin/cloudflared", "--version"], check=True)

TUNNEL_LOG = "/kaggle/working/tunnel.log"
tunnel_handle = open(TUNNEL_LOG, "w")
tunnel = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}",
     "--no-autoupdate"],
    stdout=tunnel_handle, stderr=subprocess.STDOUT,
)

PUBLIC_URL = None
deadline = time.time() + 120
while time.time() < deadline and PUBLIC_URL is None:
    text = open(TUNNEL_LOG).read()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
    if match:
        PUBLIC_URL = match.group(0)
        break
    time.sleep(3)

if not PUBLIC_URL:
    print(open(TUNNEL_LOG).read()[-4000:])
    raise SystemExit("cloudflared did not print a public URL")

print("PUBLIC URL:", PUBLIC_URL)
print()
print("Set these on the app machine (never commit them):")
print(f"LLM_PROVIDER=openai_compatible")
print(f"LLM_BASE_URL={PUBLIC_URL}/v1")
print(f"LLM_MODEL={SERVED_NAME}")
print("LLM_API_KEY=<the API_KEY you stored in Kaggle Secrets>")
print("LLM_JSON_SCHEMA_MODE=true")
print("LLM_TIMEOUT_S=120")

In [ ]:
# Cell 9 - keep-alive heartbeat.  Leave this cell running while the app is in
# use.  Stop it when the demo is over; the URL dies with the session.
import time

heartbeat = 0
try:
    while True:
        heartbeat += 1
        alive = server.poll() is None and tunnel.poll() is None
        print(f"[{heartbeat}] alive={alive} url={PUBLIC_URL}")
        tail = open(LOG_PATH).read()[-800:].strip().splitlines()
        for line in tail[-5:]:
            print("  vllm:", line)
        if not alive:
            print("A process died. Restart from Cell 5. If vLLM shows 401/403: "
                  "license not accepted or HF_TOKEN wrong.")
            break
        time.sleep(60)
except KeyboardInterrupt:
    print("heartbeat stopped")